In [2]:
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
PHOTOMETRY_LOG_URL = (
    "https://docs.google.com/spreadsheets/d/1B8KCnku1vQInKOFBuoLeuDND6CEUA4hT6qQ5zuhqvyU/"
    "export?format=csv&gid=1751471403"
)
photometry_log = pd.read_csv(PHOTOMETRY_LOG_URL)

In [4]:
photometry_log.keys()

Index(['date', 'mouse', 'fiber_1_side', 'fiber_1_area', 'fiber_1_sensor',
       'fiber_2_side', 'fiber_2_area', 'fiber_2_sensor', 'fiber_3_side',
       'fiber_3_area', 'fiber_3_sensor', 'fiber_4_side', 'fiber_4_area',
       'fiber_4_sensor', 'note'],
      dtype='str')

In [2]:
# CONFIGURATION
PHOTO_ROOT = Path("/Users/rebekahzhang/data/photometry")
PROCESSED_OUT = PHOTO_ROOT / "processed_output"
BEHAV_DIR = PHOTO_ROOT / "behav_analyzed"
FP_DIR = PHOTO_ROOT / "fp_flattened"
MATCHED_SESSIONS_CSV = PHOTO_ROOT / "matched_sessions.csv"
BEHAV_LOG_CSV = BEHAV_DIR / "sessions_photometry_exp2.csv"
PLOTS_DIR = PHOTO_ROOT / "trial_plots"

# Analysis settings
CHANNELS = ["G4", "G5", "R2", "R3"]
SIGNAL_COL = "dff_zscored"  # Use dff_zscored by default; set to "dff_filtered" for raw signal

DEFAULT_CHANNEL_COLORS = {
    "R0": "#e74c3c",
    "R1": "#c0392b",
    "R2": "#ff6b35",
    "R3": "#ff006e",
    "G4": "#27ae60",
    "G5": "#4cc9f0",
    "G6": "#2ecc71",
    "G7": "#1abc9c",
}

In [3]:
data_log = pd.read_csv(MATCHED_SESSIONS_CSV)
behav_log = pd.read_csv(BEHAV_LOG_CSV, index_col=0)
merged_log = pd.merge(data_log, behav_log, on=["mouse", "date"], how="inner")
merged_log.to_csv(PHOTO_ROOT / "sessions_dff_behav_merged.csv")

In [ ]:
session_info = merged_log.iloc[0]

In [ ]:
processed_out = PROCESSED_OUT
behav_dir = BEHAV_DIR

In [ ]:
session_id = str(session_info["session_id"])
phot = pd.read_csv(processed_out / session_id / "photometry_long.csv")

In [ ]:
phot[phot['channel']=="G4"]

In [ ]:
behav_subdir = str(session_info["dir"])
events_path = behav_dir / behav_subdir / f"events_processed_{behav_subdir}.csv"
trials_path = behav_dir / behav_subdir / f"trials_analyzed_{behav_subdir}.csv"
trials = pd.read_csv(trials_path, index_col=0)
events = pd.read_csv(events_path, index_col=0)
events_by_trial = events.groupby("session_trial_num")

In [ ]:
def assign_trials_to_photometry(
    phot: pd.DataFrame,
    trials: pd.DataFrame
) -> pd.DataFrame:
    """Align photometry to trials and compute per-sample trial time."""

    trials = trials.copy()
    session_first_start = float(trials.iloc[0]["start_time"])
    trials["start_time"] = trials["start_time"].astype(float) - session_first_start
    trials["end_time"] = trials["end_time"].astype(float) - session_first_start

    phot_sorted = phot.sort_values("t_sec").reset_index(drop=True)
    trials_sorted = trials.sort_values("start_time").reset_index(drop=True)

    merged = pd.merge_asof(
        phot_sorted,
        trials_sorted,
        left_on="t_sec",
        right_on="start_time",
        direction="backward",
    )

    merged = merged[merged["t_sec"] <= merged["end_time"]].reset_index(drop=True)
    merged["trial_time"] = merged["t_sec"] - merged["start_time"]
    return merged

In [ ]:
merged = assign_trials_to_photometry(phot, trials)

In [ ]:
trial_nums = [x for x in merged["session_trial_num"].dropna().unique()]
trial_nums = sorted(trial_nums)

In [ ]:
trial_num = trial_nums[29]

In [ ]:
sig_by_trial = merged.groupby("session_trial_num")
trial_sig = sig_by_trial.get_group(trial_num)

In [ ]:
channels_to_plot = CHANNELS
signal_col = SIGNAL_COL
channel_colors = DEFAULT_CHANNEL_COLORS

In [ ]:
trial_data = trial_sig.iloc[0]
trial_events = events_by_trial.get_group(trial_num)

In [ ]:

fig = plt.figure(figsize=(14, 6))

for ch in channels_to_plot:
    ch_df = trial_sig[trial_sig["channel"] == ch]
    if ch_df.empty:
        continue
    plt.plot(
        ch_df["trial_time"],
        ch_df[signal_col],
        label=ch,
        linewidth=1.4,
        alpha=0.85,
        color=channel_colors.get(ch, None),
    )
trial_data = trial_sig.iloc[0]
plt.axvline(0, color="green", linestyle="--", linewidth=2, alpha=0.7, label="BG start")
plt.axvline(trial_data["bg_length"], color="orange", linestyle="--", linewidth=2, alpha=0.7, label="WAIT start")
if not trial_data["miss_trial"]:
    if trial_data["reward"] > 0:
        plt.axvline(trial_data["bg_length"] + trial_data["time_waited"], color="blue", linestyle="--", linewidth=2, alpha=0.7, label="Decision")
    else:
        plt.axvline(trial_data["bg_length"] + trial_data["time_waited"], color="red", linestyle="--", linewidth=2, alpha=0.7, label="Decision")

if "key" in trial_events.columns:
    lick_events = trial_events[trial_events["key"] == "lick"]
    if len(lick_events) > 0:
        lick_times = lick_events["trial_time"].values
        for i, t in enumerate(lick_times):
            plt.axvline(t, color="grey", linestyle="-", linewidth=1, alpha=0.25, label="Licks" if i == 0 else "")

plt.xlabel("Time (s)")
plt.ylabel(signal_col)
plt.title(f"Trial {trial_data['session_trial_num']} photometry")
plt.legend(loc="upper right", fontsize=9)
plt.tight_layout()

# if save_path is not None:
#     save_path.parent.mkdir(parents=True, exist_ok=True)
#     plt.savefig(save_path, dpi=150, bbox_inches="tight")
#     plt.close(fig)
# else:
#     plt.show()